# 05 — Backtest on the test period (2025 H2)

The strategy was **frozen and committed before this period was opened** (`config/strategy.yaml`). This
notebook evaluates every frozen signal **once** on 1 Jul – 31 Dec 2025. Nothing below feeds back into the
strategy.

* Models are refit on train + validation exactly as at freeze time; positions use the saved rules and cut-offs.
* P&L = position × (RT − DA) − $0.50 × |position| per hour, 1 MW. This is a research simplification:
  price-taker, virtuals always clear, hourly settlement, one flat cost for fees and uplift.

| Signal | What it does |
|---|---|
| S0 Flat | no position (zero line) |
| S1 Always long | long the spread every hour (≈ DEC) |
| S2 Always short | short the spread every hour (≈ INC); base-rate benchmark |
| S3 Rule-based | short, flat in High/Extreme forecast load and HE17–21 |
| **S4 Primary** | **short when P(RT > DA) < 0.30** (long above 0.70: never reached) |
| S5 Risk-filtered short | short when predicted risk is below the 40th training percentile |
| S6 Risk-sized short | short with size min(1, $2.43 / predicted \|spread\|) |

In [ ]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from src.config import load_config
from src.analysis import plots as pl
from src.analysis.spreads import daily_block_bootstrap_mean_ci
from src.backtest.run import run
from src.backtest.metrics import breakdown
from src.backtest.engine import daily_pnl
from src.models.selected import predict_selected
from src.models.split import usable
from src.models.common import feature_spec

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
pl.apply_style()
FIG = load_config()["paths"]["figures"]
r = run("test")
rows = r.rows
NOTE = f"PJM Data Miner 2. Test period 1 Jul – 31 Dec 2025, strategy frozen 2026-11-08; 1 MW per hour, $0.50/MWh cost."
print(f"{len(rows):,} hours | {rows.market_date.min().date()} -> {rows.market_date.max().date()}")

## 1. Headline results

In [ ]:
tbl = r.summary_table()
tbl[["net_pnl", "pnl_per_mwh", "coverage", "hit_rate", "sharpe_like", "max_drawdown", "worst_day",
     "profit_factor", "mean_daily_pnl", "ci95_low", "ci95_high"]]

In [ ]:
daily = {s: daily_pnl(r.pnl[s], rows["market_date"]) for s in r.pnl if s != "S0_flat"}
labels = {"S1_always_long": "Always long", "S2_always_short": "Always short", "S3_rule_based": "Rule-based",
          "S4_prob_threshold": "Primary (S4)", "S5_risk_filtered_short": "Risk-filtered", "S6_risk_sized_short": "Risk-sized"}
pl.cumulative_pnl(daily, {"S4_prob_threshold": pl.BLUE, "S2_always_short": pl.ORANGE},
    "Short-side strategies made money in 2025 H2; the primary strategy trailed always-short",
    "Cumulative net P&L of every frozen signal on the test period (1 MW per hour, net of $0.50/MWh).",
    FIG / "13_test_cumulative_pnl.png", NOTE, labels)

In [ ]:
pl.drawdown_plot({"Primary (S4)": daily["S4_prob_threshold"], "Always short": daily["S2_always_short"]},
    {"Primary (S4)": pl.BLUE, "Always short": pl.ORANGE},
    "The primary strategy's worst drawdown was smaller: $1.9k vs $3.2k for always-short",
    "Drawdown from the running peak of cumulative net P&L, test period.", FIG / "14_test_drawdown.png", NOTE)

## 2. Was it the model or the market?

In [ ]:
m, lo, hi = daily_block_bootstrap_mean_ci(rows)
print(f"Test-period mean spread: {m:.2f} $/MWh (95% CI {lo:.2f} .. {hi:.2f}); median {rows.da_rt_spread.median():.2f}; "
      f"RT > DA in {(rows.da_rt_spread > 0).mean():.0%} of hours")
print("For comparison, the 18-month selection period averaged -0.28 $/MWh (95% CI -1.47 .. +1.03).")
traded = r.positions["S4_prob_threshold"] != 0
print(f"Primary strategy: hours traded average {rows.loc[traded, 'da_rt_spread'].mean():.2f}, "
      f"hours skipped average {rows.loc[~traded, 'da_rt_spread'].mean():.2f}")

In [ ]:
spec = feature_spec(); ok = usable(rows, spec.columns)
pr = predict_selected(r.models, rows[ok], spec)
y, s = rows.loc[ok, "target_rt_above_da"].to_numpy(), rows.loc[ok, "da_rt_spread"].to_numpy()
pd.Series({"direction ROC-AUC": roc_auc_score(y, pr["p_rt_above_da"]),
           "direction accuracy (p > 0.5)": ((pr["p_rt_above_da"] > 0.5).astype(int) == y).mean(),
           "always 'RT < DA' accuracy": 1 - y.mean(),
           "risk Spearman with |spread|": pd.Series(pr["pred_risk"].to_numpy()).corr(pd.Series(np.abs(s)), method="spearman")})

## 3. Where did the P&L come from?

In [ ]:
month = rows["market_date"].dt.strftime("%b")
by_month = pd.DataFrame({"Primary (S4)": breakdown(r.pnl["S4_prob_threshold"], r.positions["S4_prob_threshold"], month)["net_pnl"],
                         "Always short": breakdown(r.pnl["S2_always_short"], r.positions["S2_always_short"], month)["net_pnl"]})
by_month = by_month.reindex(["Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
pl.grouped_bars(by_month, {"Primary (S4)": pl.BLUE, "Always short": pl.ORANGE}, "Net P&L ($)",
    "Both lost money in September; the primary sat out most of August",
    "Monthly net P&L on the test period (1 MW per hour, net of $0.50/MWh).", FIG / "15_test_monthly_pnl.png", NOTE)
display(by_month)
display(breakdown(r.pnl["S4_prob_threshold"], r.positions["S4_prob_threshold"], month).reindex(["Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]))

In [ ]:
hb = pd.cut(rows["hour_ending"], [0, 6, 10, 16, 21, 24], labels=["HE1–6", "HE7–10", "HE11–16", "HE17–21", "HE22–24"])
per_mwh = pd.DataFrame({"Primary (S4)": breakdown(r.pnl["S4_prob_threshold"], r.positions["S4_prob_threshold"], hb)["pnl_per_mwh"],
                        "Always short": breakdown(r.pnl["S2_always_short"], r.positions["S2_always_short"], hb)["pnl_per_mwh"]})
per_mwh = per_mwh.reindex(["HE1–6", "HE7–10", "HE11–16", "HE17–21", "HE22–24"])
pl.grouped_bars(per_mwh, {"Primary (S4)": pl.BLUE, "Always short": pl.ORANGE}, "Net P&L per MWh traded ($)",
    "The evening peak earned the most per MWh for both",
    "Net P&L per MWh by hour block, test period.", FIG / "16_test_pnl_by_hour.png", NOTE, fmt=lambda v: pl.money(v, 0))
for s in ("S4_prob_threshold", "S2_always_short"):
    print(s); display(breakdown(r.pnl[s], r.positions[s], hb).reindex(["HE1–6", "HE7–10", "HE11–16", "HE17–21", "HE22–24"]))

In [ ]:
for s in ("S4_prob_threshold", "S2_always_short"):
    print(s); display(breakdown(r.pnl[s], r.positions[s], rows["fc_load_regime"].astype(str)).reindex(["Low", "Normal", "High", "Extreme"]))

In [ ]:
d4, d2 = daily["S4_prob_threshold"], daily["S2_always_short"]
pd.DataFrame({
    "net P&L": [d4.sum(), d2.sum()],
    "without best 5 days": [d4.sum() - d4.nlargest(5).sum(), d2.sum() - d2.nlargest(5).sum()],
    "share from best 5 days": [d4.nlargest(5).sum() / d4.sum(), d2.nlargest(5).sum() / d2.sum()],
    "worst 5 days": [d4.nsmallest(5).sum(), d2.nsmallest(5).sum()],
}, index=["Primary (S4)", "Always short"])

## Summary — test period 2025 H2 (evaluated once, frozen strategy)

| | Primary (S4) | Always short (S2) | Always long (S1) |
|---|---|---|---|
| Net P&L (1 MW, after $0.50/MWh) | **+$5,751** | +$9,799 | −$14,216 |
| P&L per MWh traded | **$2.68** | $2.22 | −$3.22 |
| Hours traded | 49% | 100% | 100% |
| Hit rate | 70% | 68% | 27% |
| Sharpe-like (daily) | 2.6 | 3.3 | −4.8 |
| Max drawdown | −$1,877 | −$3,225 | −$14,361 |
| 95% CI of mean daily P&L | −$6 … +$71 | −$5 … +$109 | −$133 … −$19 |

**What happened**
1. **The market did most of the work.** RT cleared below DA on average by **$2.72/MWh** in 2025 H2 (95% CI −$4.58 to −$0.87), against −$0.28 in the 18-month selection period. Every short-side strategy made money and always-long lost heavily. The expectation written down before testing, roughly break-even, was wrong because the period favoured shorts, not because the model improved.
2. **The primary strategy added modest selectivity, not a standalone edge.** It earned more per MWh than always-short ($2.68 vs $2.22, and more in 4 of 5 hour blocks). The hours it traded averaged −$3.17 against −$2.29 for the hours it skipped, and its worst drawdown was smaller. But it made *less in total* than always-short, its Sharpe-like ratio was lower, and **its mean daily P&L is not statistically different from zero**. It also lost money in September.
3. **Profits are concentrated.** 70% of the primary strategy's P&L came from its best 5 days (53% for always-short). Without them it made $1,723.
4. **The models behaved as Phase 4 predicted.** Risk ranking held up and improved (Spearman 0.62). Direction stayed weak (AUC 0.54; accuracy equal to always guessing "RT < DA"). The risk-filtered short (S5) lost again (−$238). Risk-sized shorting (S6) was small and positive (+$1,160).

**Conclusion.** Out of sample, with public pre-auction information, the evidence does not support a robust DA/RT spread edge at Western Hub beyond the market's own drift. The frozen strategy's positive result mostly reflects a period when RT ran persistently below DA. The direction model's contribution was better hour selection at lower volume. Phase 7 tests how sensitive these conclusions are to costs, thresholds, sub-periods and features; the 2026 holdout is then run once, unchanged.